# Traffic Signal OpenEnv: LLM-Based Reinforcement Learning Pipeline

This notebook implements a hardened RL fine-tuning pipeline for hierarchical traffic control.

In [ ]:
!pip install -q trl unsloth wandb matplotlib requests numpy datasets

In [ ]:
import os
import gc
import json
import re
import time
import random
import requests
import numpy as np
import torch
import wandb
from datasets import Dataset
from unsloth import FastLanguageModel, PatchFastRL
from trl import GRPOTrainer, GRPOConfig


In [ ]:
PatchFastRL("GRPO", FastLanguageModel)
print("PatchFastRL applied.")


In [ ]:
ENV_URL = "https://guuru-dev-traffic-signal-openenv-2.hf.space"
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
print("ENV_URL:", ENV_URL)

try:
    wandb.init(project="traffic-signal-rl", name="full-run-1")
    print("WandB initialized.")
except Exception as e:
    print(f"WandB init failed (non-fatal): {e}")
    wandb = None

r = requests.get(f"{ENV_URL}/health", timeout=30)
assert r.status_code == 200, f"Space not healthy: {r.status_code} {r.text}"
print("Space status:", r.json()["status"])
print("Task count:", r.json()["task_count"])
print("Ready to load model.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Seeds set to {SEED}.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
train_dataset = Dataset.from_dict({
    "prompt": [
        "You are a traffic controller. Output a JSON object with keys "
        "'local_actions' (dict mapping NW/NE/SW/SE to one of KEEP/SWITCH/"
        "PHASE_0/PHASE_1/PHASE_2/PHASE_3) and 'central_action' (empty dict)."
    ] * 20
})
print("Dataset size:", len(train_dataset))

def safe_post(url, action, retries=16, timeout=60):
    for attempt in range(retries):
        try:
            r = requests.post(url, json=action, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                wait = min(45, 2 * (attempt + 1)) + random.uniform(0.0, 1.0)
                print(
                    f"HTTP {r.status_code}. Waiting {wait:.1f}s "
                    f"(attempt {attempt+1}/{retries})"
                )
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            wait = min(30, 2 + attempt)
            print(
                f"Network/timeout error on attempt {attempt+1}/{retries}. "
                f"Retrying in {wait}s..."
            )
            time.sleep(wait)
    raise RuntimeError(f"Failed after {retries} retries: {url}")

def parse_action(completion):
    valid = {"KEEP", "SWITCH", "PHASE_0", "PHASE_1", "PHASE_2", "PHASE_3"}
    base = {"NW": "KEEP", "NE": "KEEP", "SW": "KEEP", "SE": "KEEP"}

    try:
        action = json.loads(completion)
        if isinstance(action, dict):
            raw_local = action.get("local_actions", {})
            clean_local = {}
            for key in ("NW", "NE", "SW", "SE"):
                raw_val = str(raw_local.get(key, "KEEP")).upper().strip() if isinstance(raw_local, dict) else "KEEP"
                clean_local[key] = raw_val if raw_val in valid else "KEEP"
            return {"local_actions": clean_local, "central_action": {}}
    except Exception:
        pass

    try:
        match = re.search(r'\{.*\}', completion, re.DOTALL)
        if match:
            return parse_action(match.group())
    except Exception:
        pass

    return {"local_actions": base, "central_action": {}}

# Anti-reward-hacking properties:
# 1. Deterministic seeded transitions
# 2. 6 independent rubric components
# 3. Reward clipped to [-1.0, 1.0]
# 4. total_priority_budget constraint prevents all-boost exploitation
# 5. Episode-level final_score prevents short-term gaming
def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for episode, (prompt, completion) in enumerate(zip(prompts, completions), start=1):

        safe_post(f"{ENV_URL}/reset",
                  {"task_id": "hard_multi", "central_enabled": True})

        action = parse_action(completion)

        episode_reward = 0.0
        done = False
        step_count = 0
        info = {}
        latency_ms = 0.0

        while not done and step_count < 100:
            t0 = time.time()
            try:
                step_res = safe_post(f"{ENV_URL}/step", action)
            except requests.HTTPError as exc:
                if getattr(exc.response, "status_code", None) == 422:
                    action = {
                        "local_actions": {"NW": "KEEP", "NE": "KEEP", "SW": "KEEP", "SE": "KEEP"},
                        "central_action": {},
                    }
                    step_res = safe_post(f"{ENV_URL}/step", action)
                else:
                    raise
            latency_ms = (time.time() - t0) * 1000
            data = step_res.json()
            info = data.get("info", {})
            episode_reward += float(data.get("reward", 0.0))
            done = data.get("done", False)
            step_count += 1
            time.sleep(0.05)

        log_data = {
            "episode_reward": episode_reward,
            "mean_queue": info.get("mean_queue", 0.0),
            "final_score": info.get("final_score", 0.0),
            "throughput": info.get("throughput", 0.0),
            "step_count": step_count,
            "step_latency_ms": latency_ms,
        }
        if wandb:
            wandb.log(log_data)

        if episode % 2 == 0:
            print(f"\n=== Episode {episode} ===")
            print(f"  Reward   : {episode_reward:.4f}")
            print(f"  Score    : {info.get('final_score', 'N/A')}")
            print(f"  Queue    : {info.get('mean_queue', 'N/A')}")
            print(f"  Steps    : {step_count}")
            print(f"  Latency  : {latency_ms:.1f}ms")
            print(f"  Action   : {action}")
            # WARNING: if action is always identical across episodes,
            # reward hacking may be occurring. Stop and inspect.

        rewards.append(episode_reward)
        time.sleep(0.2)

    return rewards

print("Reward function ready.")

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Precision config -> bf16={use_bf16}, fp16={use_fp16}")

training_args = GRPOConfig(
    output_dir="./outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    max_steps=100,
    max_prompt_length=512,
    max_completion_length=128,
    num_generations=4,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="wandb",
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    logging_steps=1,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
torch.cuda.empty_cache()
gc.collect()
print("Trainer ready. Launching training...")

checkpoint_paths = []
if os.path.exists("./outputs"):
    checkpoint_paths = [
        os.path.join("./outputs", name)
        for name in os.listdir("./outputs")
        if name.startswith("checkpoint-") and os.path.isdir(os.path.join("./outputs", name))
    ]
checkpoint_paths = sorted(checkpoint_paths, key=lambda path: int(path.rsplit("-", 1)[-1]))

if checkpoint_paths:
    latest_checkpoint = checkpoint_paths[-1]
    print("Resuming from:", latest_checkpoint)
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("Starting fresh run")
    trainer.train()

print("\nTraining complete.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# NEVER use merge-and-unload on a 4-bit model — corrupts weights
model.save_pretrained("outputs/traffic-lora")
tokenizer.save_pretrained("outputs/traffic-lora")
print("Adapter weights saved to outputs/traffic-lora")
# Optional push:
# model.push_to_hub("Guuru-DEV/traffic-signal-agent", token=os.getenv("HF_TOKEN"))
# tokenizer.push_to_hub("Guuru-DEV/traffic-signal-agent", token=os.getenv("HF_TOKEN"))


In [ ]:
if wandb:
    wandb.finish()
print("WandB run complete.")
